# CME Futures: Gradient Boosting

This notebook fits the declared LightGBM configurations for both return horizons. The grid varies
tree capacity and regression loss while the shared runner keeps folds, preprocessing, target
scaling, and checkpoint publication consistent. Huber thresholds are resolved after target
scaling, so the robust loss has the intended scale.

Every scheduled tree checkpoint is a separate prediction configuration. IC describes ranking
quality. The equal-weight validation backtests in `13_backtest` retain every checkpoint and select
by Sharpe.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

In [1]:
"""Fit the declared CME futures gradient-boosting population."""

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    model_request_catalog,
    open_study,
    product_universe_table,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_catalog,
    run_resolved_model_requests,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_REDUCTIONS: dict = {}
# The population hash this run replaces, read from the registry and set by a person. A
# first population takes None; a re-run whose membership has changed is refused without
# the hash it supersedes, and the refusal names the value required.
SUPERSEDES_POPULATION: str | None = None

## Declared requests

Configuration names and label horizons remain visible as Polars rows. Checkpoint schedules are
part of each resolved request and therefore part of prediction identity.

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
requests = model_request_catalog("gbm", labels=ALL_LABELS)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    preview_reductions=PREVIEW_REDUCTIONS,
)
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [4]:
resolved_model_plan(resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,date,date,i64,str,str
"""gbm""","""fwd_ret_21d""","""default_huber""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,10,"""canonical""","""350dfeafa98a"""
"""gbm""","""fwd_ret_21d""","""default_mae""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,10,"""canonical""","""3557f7289ab6"""
"""gbm""","""fwd_ret_21d""","""default_mse""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,10,"""canonical""","""d662632b8750"""
"""gbm""","""fwd_ret_21d""","""leaves_15_huber""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,10,"""canonical""","""fa2dc5f3d334"""
"""gbm""","""fwd_ret_21d""","""leaves_15_mae""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,10,"""canonical""","""d62d32af914b"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""gbm""","""fwd_ret_5d""","""leaves_63_mae""","""regression""",69,30,38262,5,2019-01-03,2023-12-21,10,"""canonical""","""789500821673"""
"""gbm""","""fwd_ret_5d""","""leaves_63_mse""","""regression""",69,30,38262,5,2019-01-03,2023-12-21,10,"""canonical""","""22a7d5b77c84"""
"""gbm""","""fwd_ret_5d""","""leaves_7_huber""","""regression""",69,30,38262,5,2019-01-03,2023-12-21,10,"""canonical""","""c42651364e25"""


## Execute and validate

The runner persists fitted trees and prediction shards per fold. Restart reuses a fold only after
both files and their digests validate. Publication requires the full eligible key set.

In [5]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_catalog(
        study,
        requests,
        population_name="cme-gbm-validation-v1",
        resolved_requests=resolved,
        supersedes=SUPERSEDES_POPULATION,
    )
else:
    if WORKSPACE is None or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

      fold 0: training n_train=60,680 n_val=7,518 trees=500 num_leaves=? obj=regression


      fold 0: done in 1s


      fold 1: training n_train=60,337 n_val=7,684 trees=500 num_leaves=? obj=regression


      fold 1: done in 1s


      fold 2: training n_train=60,044 n_val=7,676 trees=500 num_leaves=? obj=regression


...

      fold 2: done in 1s


      fold 3: training n_train=59,210 n_val=7,692 trees=500 num_leaves=63 obj=huber


      fold 3: done in 2s


      fold 4: training n_train=58,325 n_val=7,692 trees=500 num_leaves=63 obj=huber


      fold 4: done in 2s


In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("label", "config_name", "checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("gradient-boosting execution returned a partial prediction")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""gbm""","""fwd_ret_21d""","""default_huber""","""iteration""",50,"""canonical""",true,"""350dfeafa98a""","""51b48e8c0d4b"""
"""gbm""","""fwd_ret_21d""","""default_huber""","""iteration""",100,"""canonical""",true,"""350dfeafa98a""","""0b8388a8b7dc"""
"""gbm""","""fwd_ret_21d""","""default_huber""","""iteration""",150,"""canonical""",true,"""350dfeafa98a""","""3cc914b04b5c"""
"""gbm""","""fwd_ret_21d""","""default_huber""","""iteration""",200,"""canonical""",true,"""350dfeafa98a""","""e37d5a71f4ca"""
"""gbm""","""fwd_ret_21d""","""default_huber""","""iteration""",250,"""canonical""",true,"""350dfeafa98a""","""aed8095d2533"""
…,…,…,…,…,…,…,…,…
"""gbm""","""fwd_ret_5d""","""leaves_7_mse""","""iteration""",300,"""canonical""",true,"""a3d5ecde7375""","""1aa290eab323"""
"""gbm""","""fwd_ret_5d""","""leaves_7_mse""","""iteration""",350,"""canonical""",true,"""a3d5ecde7375""","""33870c595f82"""
"""gbm""","""fwd_ret_5d""","""leaves_7_mse""","""iteration""",400,"""canonical""",true,"""a3d5ecde7375""","""57257496d2cf"""
